# 02 – Modeling

**Goal:** Build the preprocessing pipeline, compare several models, tune
their hyperparameters and save the best pipeline.

- **Input:** `data/clean.csv` (created by `01_eda.ipynb`)
- **Output:** `models/personality_pipeline.joblib` (not committed to Git)
- **Main metric:** `f1_macro`
- **Random state:** `42` everywhere (taken from `src/config.py`)

> Run `01_eda.ipynb` first. Run this notebook from the `notebooks/`
> folder.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# The notebook lives in notebooks/, so the project root is made importable
sys.path.insert(0, str(Path.cwd().parent))

from src import config  # noqa: E402

print(f"Random state: {config.RANDOM_STATE}")

Random state: 42


## 2. Load the cleaned data and split it

In [2]:
clean_df = pd.read_csv(config.CLEAN_DATA_PATH)
print(f"Rows: {clean_df.shape[0]:,} | Columns: {clean_df.shape[1]}")

# The feature order is fixed by the config and reused by the app later
X = clean_df[config.FEATURE_COLUMNS]
y = clean_df[config.TARGET_COLUMN]

Rows: 19,514 | Columns: 23


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=config.TEST_SIZE,
    stratify=y,  # keep the class proportions in both parts
    random_state=config.RANDOM_STATE,
)

print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")

Train: 15,611 rows | Test: 3,903 rows


In [4]:
class_share = pd.DataFrame(
    {
        "full_%": y.value_counts(normalize=True) * 100,
        "train_%": y_train.value_counts(normalize=True) * 100,
        "test_%": y_test.value_counts(normalize=True) * 100,
    }
).round(2)
print(class_share)

# The stratified split must preserve the class proportions closely
assert (class_share["train_%"] - class_share["full_%"]).abs().max() < 0.1
assert (class_share["test_%"] - class_share["full_%"]).abs().max() < 0.1

                 full_%  train_%  test_%
target                                  
Moderate          42.84    42.84   42.84
Resilient         31.28    31.28   31.28
Overcontroller    14.41    14.41   14.40
Undercontroller   11.48    11.48   11.48


The data is split once into **80 % training** and **20 % test** data.
The split is **stratified**, so every class has (almost) the same share in
both parts. This is important because the classes are imbalanced.

The test set is only used for the final evaluation. Model selection and
hyperparameter tuning are done with cross-validation on the training
data.

## 3. Preprocessing sanity check

Before any model is trained, we check that the preprocessing behaves as
intended: it must be fitted on the **training data only** (no data leakage)
and it must be able to handle raw input with exactly the columns defined in
`config.FEATURE_COLUMNS`.

In [5]:
import numpy as np

from src.preprocessing import build_preprocessor

preprocessor = build_preprocessor()

# Fit on the training data only, then apply the fitted step to both parts
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f"Train shape: {X_train_transformed.shape}")
print(f"Test shape:  {X_test_transformed.shape}")
print(f"Features:    {list(feature_names)}")

Train shape: (15611, 26)
Test shape:  (3903, 26)
Features:    ['numeric__N1', 'numeric__N2', 'numeric__N3', 'numeric__N4', 'numeric__N5', 'numeric__N6', 'numeric__N7', 'numeric__N8', 'numeric__N9', 'numeric__N10', 'numeric__E1', 'numeric__E3', 'numeric__E4', 'numeric__E5', 'numeric__E7', 'numeric__E9', 'numeric__E10', 'numeric__A4', 'numeric__C4', 'numeric__age', 'categorical__gender_Female', 'categorical__gender_Male', 'categorical__gender_Other', 'categorical__hand_Both', 'categorical__hand_Left', 'categorical__hand_Right']


In [6]:
# 19 questions + age + 3 gender dummies + 3 hand dummies
assert X_train_transformed.shape[1] == 26
assert not np.isnan(X_train_transformed).any()
assert not np.isnan(X_test_transformed).any()

# The input columns (names and order) must match the config exactly
assert list(preprocessor.feature_names_in_) == config.FEATURE_COLUMNS

# The raw input must have numeric dtypes for the numeric columns
assert all(pd.api.types.is_numeric_dtype(X_train[column])
           for column in config.NUMERIC_COLUMNS)
print("Structure checks passed.")

Structure checks passed.


In [7]:
# Scaling: the fitted scaler centers the TRAINING data exactly ...
train_means = X_train_transformed[:, :20].mean(axis=0)
train_stds = X_train_transformed[:, :20].std(axis=0)
assert np.allclose(train_means, 0, atol=1e-8)
assert np.allclose(train_stds, 1, atol=1e-8)

# ... but not the test data, which proves that nothing was fitted on it
test_means = X_test_transformed[:, :20].mean(axis=0)
assert not np.allclose(test_means, 0, atol=1e-8)
print(f"Largest |mean| in the test data: {np.abs(test_means).max():.3f}")

Largest |mean| in the test data: 0.036


In [8]:
# One-hot encoding: exactly one dummy is active per feature group
gender_dummies = X_train_transformed[:, 20:23]
hand_dummies = X_train_transformed[:, 23:26]
assert (gender_dummies.sum(axis=1) == 1).all()
assert (hand_dummies.sum(axis=1) == 1).all()

# An unseen category must not raise an error (all dummies stay 0)
unseen_row = X_train.iloc[[0]].copy()
unseen_row["gender"] = "Unknown"
unseen_transformed = preprocessor.transform(unseen_row)
assert unseen_transformed[0, 20:23].sum() == 0
print("Encoding checks passed.")

Encoding checks passed.


The preprocessing works as intended:

- 26 output features (19 questions, age, 3 gender and 3 hand dummies)
- standardized on the training data only, so no information from the test
  set leaks into the model
- unseen categories are handled gracefully
- the expected input columns match `config.FEATURE_COLUMNS` exactly, which is
  the contract between the notebooks and the Streamlit app

## 4. Baselines

Before comparing real models we need a **lower bound**. Dummy classifiers
ignore the questionnaire answers completely. Every real model must beat
them clearly, otherwise it has not learned anything.

All scores in this section are **5-fold stratified cross-validation scores
on the training data**. The test set stays untouched until the final
evaluation. All models use the same folds (see `src/evaluation.py`).

In [9]:
from sklearn.base import BaseEstimator
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

from src.evaluation import cross_validate_model
from src.preprocessing import build_preprocessor


def make_pipeline(model: BaseEstimator) -> Pipeline:
    """Combine a fresh preprocessing step with the given model.

    A new preprocessor is created for every call, so that pipelines never
    share fitted state.
    """
    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            ("model", model),
        ]
    )

In [10]:
baselines = {
    "Dummy (most frequent class)": make_pipeline(
        DummyClassifier(strategy="most_frequent")
    ),
    "Dummy (random by class share)": make_pipeline(
        DummyClassifier(strategy="stratified",
                        random_state=config.RANDOM_STATE)
    ),
}

baseline_results = pd.DataFrame(
    {name: cross_validate_model(pipeline, X_train, y_train)
     for name, pipeline in baselines.items()}
).T
baseline_results.round(3)

,cv_f1_macro,cv_f1_macro_std,cv_accuracy,train_f1_macro,fit_time_s
Dummy (most frequent class),0.150,0.000,0.428,0.15,0.029
Dummy (random by class share),0.248,0.004,0.312,0.25,0.029


**Interpretation**

- Always predicting the largest class (*Moderate*) gives an accuracy of
  about 0.43 but an `f1_macro` of only about **0.15**, because the other
  three classes are never predicted.
- Guessing randomly according to the class shares gives an `f1_macro` of
  about **0.25**.
- Every real model must therefore clearly exceed an `f1_macro` of about
  0.25. The accuracy of the dummy models also shows why accuracy alone is
  a poor metric for this imbalanced problem.